<a href="https://colab.research.google.com/github/andy1680303/machine_learnig_project_2/blob/main/%E6%9C%9F%E6%9C%AB%E5%B0%88%E6%A1%88_%E8%B3%87%E6%96%99%E8%99%95%E7%90%86_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 載入資料


In [ ]:
import os
import pandas as pd
from google.colab import drive

# 1. 掛載個人的 Google 雲端硬碟空間
drive.mount('/content/drive')

# 2. 定義專案主資料夾，以及「原始資料」專屬的子資料夾
base_path = '/content/drive/MyDrive/禮拜三機器學習課/機器學習_期末專案/'
raw_folder = base_path + 'Raw_Data/'
raw_data_path = raw_folder + 'CVD_All.csv'

# 3. 確保資料夾存在 (exist_ok=True 代表如果資料夾已經存在，就不會報錯)
os.makedirs(raw_folder, exist_ok=True)
print(f"✅ 專案資料夾架構已建立：{base_path}")

# 4. 檢查原始資料是否已下載，若無則執行下載
if not os.path.exists(raw_data_path):
    print("📥 原始資料不存在，開始下載到 Raw_Data 資料夾...")
    !gdown --id 1WwF0yjAk8fgX3-qjB4njzBFTuo6l0hQN -O "{raw_data_path}"
else:
    print("ℹ️ 原始資料已經存在，跳過下載！")

# 資料前處理

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. 從雲端硬碟讀取資料 (使用設定好的 raw_data_path 與 big5 編碼)
df = pd.read_csv(raw_data_path, encoding='big5')

# 2. 剔除無預測能力的 ID 欄位
df = df.drop(columns=['ID'])

# 3. 定義特徵矩陣 X 與目標變數 y
X = df.drop(columns=['CVD'])
y = df['CVD']

# 4. 切分訓練集與測試集 (確保病患比例在兩組中維持一致)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("========== 階段一：資料前處理 (Data Preprocessing) ==========")

# (1) 缺失值填補 (使用中位數)
print("執行缺失值填補...")
imputer = SimpleImputer(strategy='median')
X_train_pre = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test_pre = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

# (2) 資料標準化
print("執行資料標準化...")
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_pre), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_pre), columns=X.columns)

# (3) 檢查高度相關特徵 (共線性)
print("檢查共線性...")
corr_matrix = X_train_scaled.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]
if to_drop:
    print(f"因高度相關而剔除的特徵: {to_drop}")
    X_train_scaled = X_train_scaled.drop(columns=to_drop)
    X_test_scaled = X_test_scaled.drop(columns=to_drop)
else:
    print("檢查完畢，無高相關性 (>0.9) 的冗餘特徵。")

feature_names = X_train_scaled.columns

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 將資料轉成 DataFrame
df = pd.DataFrame(X_train_scaled)

# 計算相關係數矩陣
corr_matrix = df.corr()

# 畫圖
sns.clustermap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    figsize=(12, 12)
)

plt.show()

# 特徵篩選

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel, RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

print("\n========== 階段二：特徵篩選 (Feature Selection) ==========")

# [方法一] SelectKBest (單變數統計值檢驗 - ANOVA)
kbest = SelectKBest(score_func=f_classif, k=8)
kbest.fit(X_train_scaled, y_train)
features_kbest = feature_names[kbest.get_support()].tolist()
print(f"[1] SelectKBest 篩選結果: {features_kbest}")

# [方法二] SelectFromModel (隨機森林內建機制)
rf_model = RandomForestClassifier(n_estimators=50, class_weight="balanced", random_state=42, n_jobs=-1)
sfm = SelectFromModel(estimator=rf_model, threshold="mean")
sfm.fit(X_train_scaled, y_train)
features_sfm = feature_names[sfm.get_support()].tolist()
print(f"[2] SelectFromModel 篩選結果: {features_sfm}")

# [方法三] RFECV (遞迴特徵移除 + 交叉驗證)
lr_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
rfecv = RFECV(estimator=lr_model, step=1, cv=3, min_features_to_select=5, scoring='roc_auc', n_jobs=-1)
rfecv.fit(X_train_scaled, y_train)
features_rfecv = feature_names[rfecv.get_support()].tolist()
print(f"[3] RFECV 最佳特徵組合 (共 {len(features_rfecv)} 個): {features_rfecv}")

# 三種特徵資料集匯出

In [ ]:
print("\n========== 階段三與四：套用三種特徵並分資料夾匯出 ==========")

# 1. 定義三個特徵篩選專屬的子資料夾路徑
# base_path 變數已經在儲存格 1 定義過
kbest_folder = base_path + '1_SelectKBest/'
sfm_folder   = base_path + '2_SelectFromModel/'
rfecv_folder = base_path + '3_RFECV/'

# 自動建立這些子資料夾
for folder in [kbest_folder, sfm_folder, rfecv_folder]:
    os.makedirs(folder, exist_ok=True)

# ---------------------------------------------------
# [版本 A] 打包 SelectKBest 資料夾
# ---------------------------------------------------
X_train_kbest = X_train_scaled[features_kbest]
X_test_kbest = X_test_scaled[features_kbest]

# 統一命名為 X_train.csv, X_test.csv，因為資料夾名稱已經區分了方法
X_train_kbest.to_csv(kbest_folder + 'X_train.csv', index=False)
X_test_kbest.to_csv(kbest_folder + 'X_test.csv', index=False)
y_train.to_csv(kbest_folder + 'y_train.csv', index=False)
y_test.to_csv(kbest_folder + 'y_test.csv', index=False)
print(f"📁 已打包 SelectKBest 資料夾 (特徵數: {X_train_kbest.shape[1]})")

# ---------------------------------------------------
# [版本 B] 打包 SelectFromModel 資料夾
# ---------------------------------------------------
X_train_sfm = X_train_scaled[features_sfm]
X_test_sfm = X_test_scaled[features_sfm]

X_train_sfm.to_csv(sfm_folder + 'X_train.csv', index=False)
X_test_sfm.to_csv(sfm_folder + 'X_test.csv', index=False)
y_train.to_csv(sfm_folder + 'y_train.csv', index=False)
y_test.to_csv(sfm_folder + 'y_test.csv', index=False)
print(f"📁 已打包 SelectFromModel 資料夾 (特徵數: {X_train_sfm.shape[1]})")

# ---------------------------------------------------
# [版本 C] 打包 RFECV 資料夾
# ---------------------------------------------------
X_train_rfecv = X_train_scaled[features_rfecv]
X_test_rfecv = X_test_scaled[features_rfecv]

X_train_rfecv.to_csv(rfecv_folder + 'X_train.csv', index=False)
X_test_rfecv.to_csv(rfecv_folder + 'X_test.csv', index=False)
y_train.to_csv(rfecv_folder + 'y_train.csv', index=False)
y_test.to_csv(rfecv_folder + 'y_test.csv', index=False)
print(f"📁 已打包 RFECV 資料夾 (特徵數: {X_train_rfecv.shape[1]})")

print("\n🎉 處理完成！三種不同特徵組合的檔案皆已成功匯出至雲端硬碟。")

# 模型訓練

## 資料集

### selectKBest

- X_train_kbest
- X_test_kbest
- y_train
- y_test

### selectFromModel

- X_train_sfm
- X_test_sfm
- y_train
- y_test


### RFECV

- X_train_rfecv
- X_test_rfecv
- y_train
- y_test


## 訓練五種模型

### Logistic Regression 分類模型


#### KBest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=7890, max_iter=5000),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train_kbest, y_train)

best_params = grid_search.best_params_

LR_KBest_model = LogisticRegression(
    C=best_params["C"],
    random_state=7890,
    class_weight="balanced",
    n_jobs=-1,
    max_iter=5000
)

LR_KBest_model.fit(X_train_kbest, y_train)

#### sfm

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=7890, max_iter=5000),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train_sfm, y_train)

best_params = grid_search.best_params_

LR_sfm_model = LogisticRegression(
    C=best_params["C"],
    random_state=7890,
    class_weight="balanced",
    n_jobs=-1,
    max_iter=5000
)

LR_sfm_model.fit(X_train_sfm, y_train)

#### rfecv

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=7890, max_iter=5000),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train_rfecv, y_train)

best_params = grid_search.best_params_

LR_rfecv_model = LogisticRegression(
    C=best_params["C"],
    random_state=7890,
    class_weight="balanced",
    n_jobs=-1,
    max_iter=5000
)

LR_rfecv_model.fit(X_train_rfecv, y_train)

### KNN 分類模型

#### KBest

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

KNN_KBest_model = KNeighborsClassifier(
    weights="distance",
    n_jobs=-1,
    n_neighbors=5
)

KNN_KBest_model.fit(X_train_kbest, y_train)

#### sfm

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

KNN_sfm_model = KNeighborsClassifier(
    weights="distance",
    n_jobs=-1,
    n_neighbors=5
)

KNN_sfm_model.fit(X_train_sfm, y_train)

#### rfecv

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

KNN_rfecv_model = KNeighborsClassifier(
    weights="distance",
    n_jobs=-1,
    n_neighbors=5
)

KNN_rfecv_model.fit(X_train_rfecv, y_train)

### Naive Bayes 分類模型

#### KBest

In [ ]:
from sklearn.naive_bayes import GaussianNB

NB_KBest_model = GaussianNB(priors=[0.5, 0.5])

NB_KBest_model.fit(X_train_kbest, y_train)

#### sfm

In [ ]:
from sklearn.naive_bayes import GaussianNB

NB_sfm_model = GaussianNB(priors=[0.5, 0.5])

NB_sfm_model.fit(X_train_sfm, y_train)

#### rfecv

In [ ]:
from sklearn.naive_bayes import GaussianNB

NB_rfecv_model = GaussianNB(priors=[0.5, 0.5])

NB_rfecv_model.fit(X_train_rfecv, y_train)

### Decision Tree 分類模型

#### KBest

In [ ]:
from sklearn import tree

DTree_KBest_model = tree.DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=123
)

DTree_KBest_model.fit(X_train_kbest, y_train)

#### sfm

In [ ]:
from sklearn import tree

DTree_sfm_model = tree.DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=123
)

DTree_sfm_model.fit(X_train_sfm, y_train)

#### rfecv

In [ ]:
from sklearn import tree

DTree_rfecv_model = tree.DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=123
)

DTree_rfecv_model.fit(X_train_rfecv, y_train)

### XGBoost 分類模型

#### KBest

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_KBest_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=123
)

xgb_KBest_model.fit(X_train_kbest, y_train)

#### sfm

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_sfm_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=123
)

xgb_sfm_model.fit(X_train_sfm, y_train)

#### rfecv

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_rfecv_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=123
)

xgb_rfecv_model.fit(X_train_rfecv, y_train)

# 評估模型

## 評估五種模型

In [ ]:
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report

### Logistic Regression 分類模型

#### KBest

In [ ]:
# 測試集準確率
testing_performance = LR_KBest_model.score(X_test_kbest, y_test)
print(f"Logistic Regression (KBest) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    LR_KBest_model,
    X_test_kbest,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = LR_KBest_model.predict(X_test_kbest)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

plt.figure(figsize=(8, 6))
display.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### sfm

In [ ]:
# 測試集準確率
testing_performance = LR_sfm_model.score(X_test_sfm, y_test)
print(f"Logistic Regression (sfm) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    LR_sfm_model,
    X_test_sfm,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = LR_sfm_model.predict(X_test_sfm)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()


# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### rfecv

In [ ]:
# 測試集準確率
testing_performance = LR_rfecv_model.score(X_test_rfecv, y_test)
print(f"Logistic Regression (rfecv) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    LR_rfecv_model,
    X_test_rfecv,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = LR_rfecv_model.predict(X_test_rfecv)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

### KNN 分類模型

#### KBest

In [ ]:
# 測試集準確率
testing_performance = KNN_KBest_model.score(X_test_kbest, y_test)
print(f"KNN (KBest) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    KNN_KBest_model,
    X_test_kbest,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = KNN_KBest_model.predict(X_test_kbest)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### sfm

In [ ]:
# 測試集準確率
testing_performance = KNN_sfm_model.score(X_test_sfm, y_test)
print(f"KNN (sfm) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    KNN_sfm_model,
    X_test_sfm,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = KNN_sfm_model.predict(X_test_sfm)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### rfecv

In [ ]:
# 測試集準確率
testing_performance = KNN_rfecv_model.score(X_test_rfecv, y_test)
print(f"KNN (rfecv) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    KNN_rfecv_model,
    X_test_rfecv,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = KNN_rfecv_model.predict(X_test_rfecv)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

### Naive Bayes 分類模型

#### KBest

In [ ]:
# 測試集準確率
testing_performance = NB_KBest_model.score(X_test_kbest, y_test)
print(f"NB (KBest) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    NB_KBest_model,
    X_test_kbest,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = NB_KBest_model.predict(X_test_kbest)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### sfm

In [ ]:
# 測試集準確率
testing_performance = NB_sfm_model.score(X_test_sfm, y_test)
print(f"NB (sfm) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    NB_sfm_model,
    X_test_sfm,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = NB_sfm_model.predict(X_test_sfm)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### rfecv

In [ ]:
# 測試集準確率
testing_performance = NB_rfecv_model.score(X_test_rfecv, y_test)
print(f"NB (rfecv) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    NB_rfecv_model,
    X_test_rfecv,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = NB_rfecv_model.predict(X_test_rfecv)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

### Decision Tree 分類模型

#### KBest

In [ ]:
# 測試集準確率
testing_performance = DTree_KBest_model.score(X_test_kbest, y_test)
print(f"DTree (KBest) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    DTree_KBest_model,
    X_test_kbest,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = DTree_KBest_model.predict(X_test_kbest)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### sfm

In [ ]:
# 測試集準確率
testing_performance = DTree_sfm_model.score(X_test_sfm, y_test)
print(f"DTree (sfm) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    DTree_sfm_model,
    X_test_sfm,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = DTree_sfm_model.predict(X_test_sfm)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### rfecv

In [ ]:
# 測試集準確率
testing_performance = DTree_rfecv_model.score(X_test_rfecv, y_test)
print(f"DTree (rfecv) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    DTree_rfecv_model,
    X_test_rfecv,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = DTree_rfecv_model.predict(X_test_rfecv)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

### XGBoost 分類模型

#### KBest

In [ ]:
# 測試集準確率
testing_performance = xgb_KBest_model.score(X_test_kbest, y_test)
print(f"xgb (KBest) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    xgb_KBest_model,
    X_test_kbest,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = xgb_KBest_model.predict(X_test_kbest)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### sfm

In [ ]:
# 測試集準確率
testing_performance = xgb_sfm_model.score(X_test_sfm, y_test)
print(f"xgb (sfm) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    xgb_sfm_model,
    X_test_sfm,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = xgb_sfm_model.predict(X_test_sfm)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

#### rfecv

In [ ]:
# 測試集準確率
testing_performance = xgb_rfecv_model.score(X_test_rfecv, y_test)
print(f"xgb (rfecv) Testing Performance: {testing_performance}")

# ROC curve
RocCurveDisplay.from_estimator(
    xgb_rfecv_model,
    X_test_rfecv,
    y_test
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.legend()
plt.show()

# 建立混淆矩陣
y_test_pred = xgb_rfecv_model.predict(X_test_rfecv)
cm = confusion_matrix(y_test, y_test_pred)

# 顯示混淆矩陣
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
)

display.plot()

# 分類報告
report = classification_report(
    y_test,
    y_test_pred,
)

print(report)

## SHAP 模型解釋

In [ ]:
import shap

### Logistic Regression 分類模型

#### KBest

In [ ]:
X_background = shap.sample(X_train_kbest, 100, random_state=123)
X_test_sample = shap.sample(X_test_kbest, 100, random_state=123)

explainer = shap.Explainer(
    LR_KBest_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### sfm

In [ ]:
X_background = shap.sample(X_train_sfm, 100, random_state=123)
X_test_sample = shap.sample(X_test_sfm, 100, random_state=123)

explainer = shap.Explainer(
    LR_sfm_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### rfecv

In [ ]:
X_background = shap.sample(X_train_rfecv, 100, random_state=123)
X_test_sample = shap.sample(X_test_rfecv, 100, random_state=123)

explainer = shap.Explainer(
    LR_rfecv_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

### KNN 分類模型

#### KBest

In [ ]:
X_background = shap.sample(X_train_kbest, 100, random_state=123)
X_test_sample = shap.sample(X_test_kbest, 100, random_state=123)

explainer = shap.Explainer(
    KNN_KBest_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### sfm

In [ ]:
X_background = shap.sample(X_train_sfm, 100, random_state=123)
X_test_sample = shap.sample(X_test_sfm, 100, random_state=123)

explainer = shap.Explainer(
    KNN_sfm_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### rfecv

In [ ]:
X_background = shap.sample(X_train_rfecv, 100, random_state=123)
X_test_sample = shap.sample(X_test_rfecv, 100, random_state=123)

explainer = shap.Explainer(
    KNN_rfecv_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

### Naive Bayes 分類模型

#### KBest

In [ ]:
X_background = shap.sample(X_train_kbest, 100, random_state=123)
X_test_sample = shap.sample(X_test_kbest, 100, random_state=123)

explainer = shap.Explainer(
    NB_KBest_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### sfm

In [ ]:
X_background = shap.sample(X_train_sfm, 100, random_state=123)
X_test_sample = shap.sample(X_test_sfm, 100, random_state=123)

explainer = shap.Explainer(
    NB_sfm_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### rfecv

In [ ]:
X_background = shap.sample(X_train_rfecv, 100, random_state=123)
X_test_sample = shap.sample(X_test_rfecv, 100, random_state=123)

explainer = shap.Explainer(
    NB_rfecv_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

### Decision Tree 分類模型

#### KBest

In [ ]:
X_background = shap.sample(X_train_kbest, 100, random_state=123)
X_test_sample = shap.sample(X_test_kbest, 100, random_state=123)

explainer = shap.Explainer(
    DTree_KBest_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### sfm

In [ ]:
X_background = shap.sample(X_train_sfm, 100, random_state=123)
X_test_sample = shap.sample(X_test_sfm, 100, random_state=123)

explainer = shap.Explainer(
    DTree_sfm_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### rfecv

In [ ]:
X_background = shap.sample(X_train_rfecv, 100, random_state=123)
X_test_sample = shap.sample(X_test_rfecv, 100, random_state=123)

explainer = shap.Explainer(
    DTree_rfecv_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

### XGBoost 分類模型

#### KBest

In [ ]:
X_background = shap.sample(X_train_kbest, 100, random_state=123)
X_test_sample = shap.sample(X_test_kbest, 100, random_state=123)

explainer = shap.Explainer(
    xgb_KBest_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### sfm

In [ ]:
X_background = shap.sample(X_train_sfm, 100, random_state=123)
X_test_sample = shap.sample(X_test_sfm, 100, random_state=123)

explainer = shap.Explainer(
    xgb_sfm_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)

#### rfecv

In [ ]:
X_background = shap.sample(X_train_rfecv, 100, random_state=123)
X_test_sample = shap.sample(X_test_rfecv, 100, random_state=123)

explainer = shap.Explainer(
    xgb_rfecv_model.predict_proba,
    X_background
)

shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values = shap_values[:, :, 1], max_display=12)